In [1]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
from scipy import stats
import re
import os
from seaborn import FacetGrid
from matplotlib.lines import Line2D
import glob
from matplotlib.ticker import FormatStrFormatter

# ---------- Global style scheme ----------
# Seven metrics (functions): loss, acc, fairDP, fairEO, rel, res, priv.
# Domain hue:  perf = blue, fair = green, rel/res = red, priv = black.
#
# Line plots (Fig 2 / SE):  full-saturation hue, SOLID vs DASHED within a domain
#     loss   blue   solid       acc    blue   dashed
#     fairDP green  solid       fairEO green  dashed
#     rel    red    solid       res    red    dashed
#     priv   black  solid
#
# Box / density plots (Fig 3, 4, 7, 8 / S, SD):  MAX (255) vs HALF (128) shade
#     loss   (0,0,255)   acc    (0,0,128)
#     fairDP (0,255,0)   fairEO (0,128,0)
#     rel    (255,0,0)   res    (128,0,0)
#     priv   (0,0,0)
DOMAIN_COLORS = {
    "perf": "#0000ff",  # blue
    "fair": "#00ff00",  # green
    "rel":  "#ff0000",  # red
    "priv": "#000000",  # black
}

# Per-metric fill colors for box / density plots (max vs half shade per domain).
BOX_COLORS = {
    "loss":   "#0000ff",  # blue  255
    "acc":    "#000080",  # blue  128
    "fairDP": "#00ff00",  # green 255
    "fairEO": "#008000",  # green 128
    "rel":    "#ff0000",  # red   255
    "res":    "#800000",  # red   128
    "priv":   "#000000",  # black
}

# Human-facing labels = the function names themselves.
METRIC_LABELS = {
    "loss": "loss", "acc": "acc",
    "fairDP": "fairDP", "fairEO": "fairEO",
    "rel": "rel", "res": "res", "priv": "priv",
}


def line_style_for(method):
    """Line plot styling (Fig 2 / SE): full-saturation hue, solid/dashed within a
    domain. Keys are the *global* metric column names; 'accuracy' is kept as an
    acc alias for older CSVs that have not been re-exported with global_*_acc."""
    return {
        f"global_{method}_loss":    {"color": DOMAIN_COLORS["perf"], "linestyle": "-"},
        f"global_{method}_acc":     {"color": DOMAIN_COLORS["perf"], "linestyle": "--"},
        "accuracy":                 {"color": DOMAIN_COLORS["perf"], "linestyle": "--"},
        f"global_{method}_fair":    {"color": DOMAIN_COLORS["fair"], "linestyle": "-"},
        f"global_{method}_fair_eo": {"color": DOMAIN_COLORS["fair"], "linestyle": "--"},
        f"global_{method}_rob":     {"color": DOMAIN_COLORS["rel"],  "linestyle": "-"},
        f"global_{method}_adv":     {"color": DOMAIN_COLORS["rel"],  "linestyle": "--"},
        f"global_{method}_priv":    {"color": DOMAIN_COLORS["priv"], "linestyle": "-"},
    }


# Canonical metric order + per-client contribution column candidates.
def _contrib_candidates(method):
    return [
        ("loss",   [f"{method}_loss_contribution", "loss_contribution"]),
        ("acc",    ["acc_contribution", f"{method}_acc_contribution"]),
        ("fairDP", [f"{method}_fair_contribution"]),
        ("fairEO", [f"{method}_fair_eo_contribution"]),
        ("rel",    [f"{method}_rob_contribution"]),
        ("res",    [f"{method}_adv_contribution"]),
        ("priv",   [f"{method}_priv_contribution"]),
    ]


def contrib_columns(df, method, include_fair=True):
    """[(key, column)] for the metrics actually present in `df`, in canonical
    order (loss, acc, fairDP, fairEO, rel, res, priv). Fairness metrics are
    skipped when include_fair=False (e.g. IMDB has no sensitive attribute)."""
    out = []
    for key, cands in _contrib_candidates(method):
        if not include_fair and key in ("fairDP", "fairEO"):
            continue
        col = next((c for c in cands if c in df.columns), None)
        if col is not None:
            out.append((key, col))
    return out


def box_color_for(method):
    """Box / density palette keyed by the per-client contribution column name.
    Max vs half shade within a domain (replaces the old solid/dashed split)."""
    return {
        "loss_contribution":              BOX_COLORS["loss"],
        f"{method}_loss_contribution":    BOX_COLORS["loss"],
        "acc_contribution":               BOX_COLORS["acc"],
        f"{method}_acc_contribution":     BOX_COLORS["acc"],
        f"{method}_fair_contribution":    BOX_COLORS["fairDP"],
        f"{method}_fair_eo_contribution": BOX_COLORS["fairEO"],
        f"{method}_rob_contribution":     BOX_COLORS["rel"],
        f"{method}_adv_contribution":     BOX_COLORS["res"],
        f"{method}_priv_contribution":    BOX_COLORS["priv"],
    }


def color_box_borders(ax):
    """For 20-client plots: color box edges + whisker/cap/median lines to match
    each box's facecolor, so shrunk boxes don't look like black blobs."""
    boxes = [p for p in ax.patches if hasattr(p, "get_facecolor")]
    if not boxes:
        return
    n_lines = len(ax.lines)
    per_box = n_lines // len(boxes) if boxes else 0
    for i, patch in enumerate(boxes):
        fc = patch.get_facecolor()
        patch.set_edgecolor(fc)
        patch.set_linewidth(1.2)
        for j in range(per_box):
            idx = i * per_box + j
            if idx < n_lines:
                ax.lines[idx].set_color(fc)
                ax.lines[idx].set_markeredgecolor(fc)
                ax.lines[idx].set_markerfacecolor(fc)


Method amit nezunk:

In [2]:
#choose method 'gtg' or 'l1o'
METHOD = 'gtg'

Adathalmazok összerakása:

In [3]:
adult_data_gtg = pd.read_csv("results_adult_gtg.csv")
adultnoniid_data_gtg = pd.read_csv("results_adultnoniid_gtg.csv")
celeba_data_gtg = pd.read_csv("results_celeba_gtg.csv")
celebanoniid_data_gtg = pd.read_csv("results_celebanoniid_gtg.csv")
imdb_data_gtg = pd.read_csv("results_imdb_gtg.csv")
imdbnoniid_data_gtg = pd.read_csv("results_imdbnoniid_gtg.csv")
cifar_data_gtg = pd.read_csv("results_cifar_gtg.csv")
cifarnoniid_data_gtg = pd.read_csv("results_cifarnoniid_gtg.csv")

adult_data_loo = pd.read_csv("results_adult_l1o.csv")
adultnoniid_data_loo = pd.read_csv("results_adultnoniid_l1o.csv")
celeba_data_loo = pd.read_csv("results_celeba_l1o.csv")
celebanoniid_data_loo = pd.read_csv("results_celebanoniid_l1o.csv")
imdb_data_loo = pd.read_csv("results_imdb_l1o.csv")
imdbnoniid_data_loo = pd.read_csv("results_imdbnoniid_l1o.csv")
cifar_data_loo = pd.read_csv("results_cifar_l1o.csv")
cifarnoniid_data_loo = pd.read_csv("results_cifarnoniid_l1o.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'results_adult_gtg.csv'

In [ ]:
df_gtg = pd.concat([adult_data_gtg, adultnoniid_data_gtg, celeba_data_gtg, celebanoniid_data_gtg, imdb_data_gtg, imdbnoniid_data_gtg, cifar_data_gtg, cifarnoniid_data_gtg])
df_loo = pd.concat([adult_data_loo, adultnoniid_data_loo, celeba_data_loo, celebanoniid_data_loo, imdb_data_loo, imdbnoniid_data_loo, cifar_data_loo, cifarnoniid_data_loo])

Kontroll panel arra h leplottoljuk az ábrákat, előbb le kell futtatni a többi cellát.


In [ ]:
os.makedirs("plots", exist_ok=True)

def functions():
    #függvényhívások
    global_metric_evolutions(method=METHOD)
    imdb_boxplots_3metrics_4cli(method=METHOD)
    adult_celeba_boxplots_4metrics_4cli(method=METHOD)
    imdb_boxplots_4metrics_20cli(method=METHOD)
    adult_celeba_boxplots_4metrics_20cli(method=METHOD)
    generate_heatmaps(method=METHOD)
    distribution_scores(method=METHOD)

functions()


In [ ]:
#downloading plots in a zip
!zip -r plots.zip plots
from google.colab import files
files.download("plots.zip")

updating: plots/ (stored 0%)
updating: plots/SE_A_4_N.png (deflated 4%)
updating: plots/SE_C_4_I.png (deflated 5%)
updating: plots/H_20_I_I_l1o.png (deflated 12%)
updating: plots/S_I_4_N_l1o.png (deflated 22%)
updating: plots/SE_I_4_N.png (deflated 7%)
updating: plots/H_4_C_N_l1o.png (deflated 8%)
updating: plots/S_A_20_I_l1o.png (deflated 25%)
updating: plots/H_4_A_I_l1o.png (deflated 10%)
updating: plots/S_I_4_I_l1o.png (deflated 23%)
updating: plots/SD_4_A_N_l1o.png (deflated 10%)
updating: plots/SD_4_A_I_l1o.png (deflated 11%)
updating: plots/SE_A_4_I.png (deflated 6%)
updating: plots/H_4_I_I_l1o.png (deflated 12%)
updating: plots/SE_I_4_I.png (deflated 7%)
updating: plots/SE_I_20_N.png (deflated 5%)
updating: plots/SE_C_4_N.png (deflated 4%)
updating: plots/SD_4_I_I_l1o.png (deflated 12%)
updating: plots/SD_20_I_N_l1o.png (deflated 11%)
updating: plots/S_A_4_I_l1o.png (deflated 22%)
updating: plots/SD_20_C_I_l1o.png (deflated 13%)
updating: plots/SE_A_20_N.png (deflated 5%)
updati

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Függvények az adatok kiszűrésére:

In [ ]:
def prepare_ac_data(given_df, dataset_name, method, cl_num, parti):
    data = given_df[given_df['dataset'] == dataset_name]
    data = data[data['partition'] == parti]
    data = data[data['num_clients'] == cl_num]
    # loss, acc, fairDP, fairEO, rel, res, priv (whichever columns exist)
    metric_cols = [col for _, col in contrib_columns(data, method, include_fair=True)]
    return data.groupby(
        ['dataset', 'partition', 'num_clients', 'client_id', 'round']
    )[metric_cols].mean().reset_index()


In [ ]:
#prepares the data based on the number of clients dataset name method and partition
def prepare_i_data(given_df, dataset_name, method, cl_num, parti):
    data = given_df[given_df['dataset'] == dataset_name]
    data = data[data['partition'] == parti]
    data = data[data['num_clients'] == cl_num]
    # IMDB: loss, acc, rel, res, priv (no fairness — no sensitive attribute)
    metric_cols = [col for _, col in contrib_columns(data, method, include_fair=False)]
    return data.groupby(
        ['dataset', 'partition', 'num_clients', 'client_id', 'round']
    )[metric_cols].mean().reset_index()


Függvények amik a plotokat gyártják:

In [ ]:
def global_metric_evolutions(method):

    df = df_gtg if method == 'gtg' else df_loo

    # --- what we want to plot: loss, acc, fairDP, fairEO, rel, res, priv ---
    requested_metrics = [
        f"global_{method}_loss",
        f"global_{method}_acc",
        f"global_{method}_fair",
        f"global_{method}_fair_eo",
        f"global_{method}_rob",
        f"global_{method}_adv",
        f"global_{method}_priv",
    ]
    # Fall back to the legacy 'accuracy' column when global_*_acc is absent.
    if f"global_{method}_acc" not in df.columns and "accuracy" in df.columns:
        requested_metrics[1] = "accuracy"

    base_names   = ["adult", "celeba", "cifar", "imdb"]
    client_cases = [4, 20]
    parts        = ["iid", "noniid"]

    # Normalize / safety
    df = df.copy()
    df["dataset"]   = df["dataset"].astype(str).str.strip().str.lower()
    df["partition"] = df["partition"].astype(str).str.strip().str.lower()
    df["round"]     = pd.to_numeric(df["round"], errors="coerce")

    df["dataset_base"] = df["dataset"].str.replace("noniid", "", regex=False)
    # Eq.(2): plot loss as the e^{-L} utility (higher=better), consistent with Tab.4
    _lcol = f"global_{method}_loss"
    if _lcol in df.columns:
        df[_lcol] = np.exp(-pd.to_numeric(df[_lcol], errors="coerce"))

    available_metrics = [m for m in requested_metrics if m in df.columns]

    # Domain-grouped style: solid/dashed within same color
    line_style = line_style_for(method)

    label_map = {
        f"global_{method}_loss": "loss",
        f"global_{method}_acc": "acc",
        "accuracy": "acc",
        f"global_{method}_fair": "fairDP",
        f"global_{method}_fair_eo": "fairEO",
        f"global_{method}_rob": "rel",
        f"global_{method}_adv": "res",
        f"global_{method}_priv": "priv",
    }

    # Filename codes (CelebA replaces CIFAR; reuse 'C')
    dataset_code = {"adult": "A", "celeba": "C", "cifar": "CI", "imdb": "I"}
    part_code    = {"iid": "I", "noniid": "N"}

    # --- Plot per combination ---
    for base in base_names:
        for clients in client_cases:
            for part in parts:

                sub = df[
                    (df["dataset_base"] == base) &
                    (df["partition"] == part) &
                    (df["num_clients"] == clients)
                ].copy()

                if sub.empty:
                    continue

                metrics = [m for m in available_metrics if m in sub.columns]
                if not metrics:
                    continue

                # aggregate per round
                agg = sub.groupby("round")[metrics].agg(["mean", "std"])
                agg.columns = [f"{c[0]}_{c[1]}" for c in agg.columns]
                agg = agg.reset_index().sort_values("round")

                fig, ax = plt.subplots(figsize=(6, 5))

                ax.set_xlabel("Round", fontsize=20, labelpad=10)
                ax.set_ylabel("Metric", fontsize=20, labelpad=10)
                ax.tick_params(axis="both", labelsize=16, width=2.0, length=6)

                for m in metrics:
                    style = line_style.get(m, {"color": "#444444", "linestyle": "-"})
                    mean_col = f"{m}_mean"
                    std_col  = f"{m}_std"

                    x = agg["round"].to_numpy()
                    y = agg[mean_col].to_numpy()
                    s = np.nan_to_num(agg[std_col].to_numpy(), nan=0.0)

                    ax.errorbar(
                        x, y, yerr=s,
                        fmt=style["linestyle"],
                        color=style["color"],
                        capsize=8,
                        linewidth=4,
                        elinewidth=2,
                        markeredgewidth=1.2,
                        label="_nolegend_",
                    )

                ax.grid(True, linestyle="--", alpha=0.7)

                base_code = dataset_code.get(base, base[0].upper())
                p_code    = part_code[part]
                filename  = f"SE_{base_code}_{clients}_{p_code}.png"

                fig.tight_layout()
                fig.savefig(os.path.join("plots", filename), bbox_inches="tight")
                plt.close(fig)

    # --- Global legend (single PNG, line-style: solid/dashed, full-saturation) ---
    legend_handles = []
    legend_labels  = []

    for m in available_metrics:
        style = line_style.get(m, {"color": "#444444", "linestyle": "-"})
        h = Line2D(
            [0], [0],
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=3,
            markersize=8,
        )
        legend_handles.append(h)
        legend_labels.append(label_map.get(m, m))

    if legend_handles:
        ncol = min(len(legend_handles), 7)
        legend_fig, legend_ax = plt.subplots(figsize=(max(4, 1.1 * len(legend_handles)), 0.9))
        legend_ax.legend(legend_handles, legend_labels, ncol=ncol, fontsize=10, frameon=False, loc="center")
        legend_ax.axis("off")
        legend_fig.tight_layout()
        legend_fig.savefig(os.path.join("plots", "global_legend.png"), bbox_inches="tight")
        plt.close(legend_fig)


In [ ]:
def imdb_boxplots_3metrics_4cli(method):
    # --- ONLY IMDB DATA ---
    df = df_gtg if method == 'gtg' else df_loo

    i_4 = prepare_i_data(df, 'imdb', method, 4, 'iid')
    in_4 = prepare_i_data(df, 'imdb', method, 4, 'noniid')

    plot_data = {
        ('imdb', 'IID'): i_4,
        ('imdb', 'Non-IID'): in_4,
    }

    dataset_code = {'imdb': 'I'}
    partition_code = {'IID': 'I', 'Non-IID': 'N'}

    for (dataset, partition), df in plot_data.items():

        fig, ax = plt.subplots(figsize=(5, 4))

        ax.set_xlabel("Client", fontsize=16, labelpad=10)
        ax.set_ylabel("Metric", fontsize=16, labelpad=10)
        ax.tick_params(axis="both", labelsize=14, width=2.0, length=6)

        # loss, acc, rel, res, priv (no fairness for IMDB)
        pairs = contrib_columns(df, method, include_fair=False)
        value_vars = [col for _, col in pairs]
        palette = [BOX_COLORS[key] for key, _ in pairs]

        df_melted = df.melt(
            id_vars=['client_id'],
            value_vars=value_vars,
            var_name='type',
            value_name='value'
        )

        sns.boxplot(
            data=df_melted,
            x='client_id',
            y='value',
            hue='type',
            hue_order=value_vars,
            ax=ax,
            palette=palette,  # explicit per-box colors in canonical metric order
            showfliers=False,
            legend=False
        )

        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

        unique_clients = sorted(df_melted['client_id'].unique())
        for i in range(len(unique_clients) - 1):
            ax.axvline(i + 0.5, color='gray',
                      linestyle='-', linewidth=1, zorder=0)

        plt.tight_layout()

        num_clients = int(df['num_clients'].iloc[0])

        d_code = dataset_code['imdb']
        p_code = partition_code[partition]
        filename = f"plots/S_{d_code}_{num_clients}_{p_code}_{method}.png"

        plt.savefig(filename, dpi=300)
        plt.show()
        plt.close(fig)


In [ ]:
def adult_celeba_boxplots_4metrics_4cli(method):
    # --- ONLY ADULT AND CELEBA DATA ---
    df = df_gtg if method == 'gtg' else df_loo

    a_4  = prepare_ac_data(df, 'adult',  method, 4, 'iid')
    an_4 = prepare_ac_data(df, 'adult',  method, 4, 'noniid')
    c_4  = prepare_ac_data(df, 'celeba', method, 4, 'iid')
    cn_4 = prepare_ac_data(df, 'celeba', method, 4, 'noniid')
    ci_4  = prepare_ac_data(df, 'cifar', method, 4, 'iid')
    cin_4 = prepare_ac_data(df, 'cifar', method, 4, 'noniid')

    plot_data = {
        ('adult',  'IID'):     a_4,
        ('adult',  'Non-IID'): an_4,
        ('celeba', 'IID'):     c_4,
        ('celeba', 'Non-IID'): cn_4,
        ('cifar',  'IID'):     ci_4,
        ('cifar',  'Non-IID'): cin_4,
    }

    dataset_code = {
        'adult':  'A',
        'celeba': 'C',
        'cifar':  'CI',
        'imdb':   'I',
    }
    partition_code = {'IID': 'I', 'Non-IID': 'N'}

    for (dataset, partition), df in plot_data.items():
        fig, ax = plt.subplots(figsize=(5, 4))

        ax.set_xlabel("Client", fontsize=16, labelpad=10)
        ax.set_ylabel("Metric", fontsize=16, labelpad=10)
        ax.tick_params(axis="both", labelsize=14, width=2.0, length=6)

        # loss, acc, fairDP, fairEO, rel, res, priv (whichever exist)
        pairs = contrib_columns(df, method, include_fair=True)
        value_vars = [col for _, col in pairs]
        palette = [BOX_COLORS[key] for key, _ in pairs]

        df_melted = df.melt(
            id_vars=['client_id'],
            value_vars=value_vars,
            var_name='type',
            value_name='value'
        )

        sns.boxplot(
            data=df_melted,
            x='client_id',
            y='value',
            hue='type',
            hue_order=value_vars,
            ax=ax,
            palette=palette,
            showfliers=False,
            legend=False
        )

        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

        unique_clients = sorted(df_melted['client_id'].unique())
        for i in range(len(unique_clients) - 1):
            ax.axvline(i + 0.5, color='gray',
                      linestyle='-', linewidth=1, zorder=0)

        plt.tight_layout()

        if 'num_clients' in df.columns:
            num_clients = int(df['num_clients'].iloc[0])
        else:
            num_clients = 4

        d_code = dataset_code.get(dataset, dataset[0].upper())
        p_code = partition_code[partition]
        filename = f"plots/S_{d_code}_{num_clients}_{p_code}_{method}.png"

        plt.savefig(filename, dpi=300)
        plt.show()
        plt.close(fig)

    # --- Shared legend for the box / density figures (Fig 3, 4, 7, 8) ---
    # Max vs half shade within each domain; labels are the function names.
    legend_entries = [
        ('loss',   BOX_COLORS['loss']),
        ('acc',    BOX_COLORS['acc']),
        ('fairDP', BOX_COLORS['fairDP']),
        ('fairEO', BOX_COLORS['fairEO']),
        ('rel',    BOX_COLORS['rel']),
        ('res',    BOX_COLORS['res']),
        ('priv',   BOX_COLORS['priv']),
    ]

    handles = [
        Line2D(
            [0], [0],
            marker='s', linestyle='',
            markersize=20,
            markerfacecolor=color,
            markeredgecolor=color,
            label=label,
        )
        for label, color in legend_entries
    ]

    fig, ax = plt.subplots(figsize=(6, 1.3))
    ax.legend(
        handles=handles,
        ncol=7,
        fontsize=12,
        frameon=False,
        loc='center',
    )
    ax.axis('off')

    fig.tight_layout()
    fig.savefig("plots/S_legend.png", dpi=300, bbox_inches='tight')
    plt.close(fig)


In [ ]:
def imdb_boxplots_4metrics_20cli(method):
    df = df_gtg if method == 'gtg' else df_loo

    i_20 = prepare_i_data(df, 'imdb', method, 20, 'iid')
    in_20 = prepare_i_data(df, 'imdb', method, 20, 'noniid')

    plot_data = {
        ('imdb', 'IID'): i_20,
        ('imdb', 'Non-IID'): in_20,
    }

    dataset_code = {'imdb': 'I'}
    partition_code = {'IID': 'I', 'Non-IID': 'N'}

    for (dataset, partition), df in plot_data.items():

        fig, ax = plt.subplots(figsize=(5, 4))

        # loss, acc, rel, res, priv (no fairness for IMDB)
        pairs = contrib_columns(df, method, include_fair=False)
        value_vars = [col for _, col in pairs]
        palette = [BOX_COLORS[key] for key, _ in pairs]

        df_melted = df.melt(
            id_vars=['client_id'],
            value_vars=value_vars,
            var_name='type',
            value_name='value'
        )

        sns.boxplot(
            data=df_melted,
            x='client_id',
            y='value',
            hue='type',
            hue_order=value_vars,
            ax=ax,
            palette=palette,  # explicit per-box colors in canonical metric order
            showfliers=False,
            legend=False
        )

        # Color box borders + whisker/cap/median lines so shrunk boxes stay visible
        color_box_borders(ax)

        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_ylabel("Metric", fontsize=10)
        ax.set_xlabel("Client", fontsize=10)
        ax.tick_params(axis='x', labelsize=8)

        unique_clients = sorted(df_melted['client_id'].unique())
        for i in range(len(unique_clients) - 1):
            ax.axvline(i + 0.5, color='lightgray',
                      linestyle='--', linewidth=0.8, zorder=0)

        plt.tight_layout()

        num_clients = int(df['num_clients'].iloc[0])

        d_code = dataset_code['imdb']
        p_code = partition_code[partition]
        filename = f"plots/S_{d_code}_{num_clients}_{p_code}_{method}.png"

        plt.savefig(filename, dpi=300)
        plt.show()
        plt.close(fig)


In [ ]:
def adult_celeba_boxplots_4metrics_20cli(method):
    df = df_gtg if method == 'gtg' else df_loo

    a_20  = prepare_ac_data(df, 'adult',  method, 20, 'iid')
    an_20 = prepare_ac_data(df, 'adult',  method, 20, 'noniid')
    c_20  = prepare_ac_data(df, 'celeba', method, 20, 'iid')
    cn_20 = prepare_ac_data(df, 'celeba', method, 20, 'noniid')
    ci_20  = prepare_ac_data(df, 'cifar', method, 20, 'iid')
    cin_20 = prepare_ac_data(df, 'cifar', method, 20, 'noniid')

    plot_data = {
        ('adult',  'IID'):     a_20,
        ('adult',  'Non-IID'): an_20,
        ('celeba', 'IID'):     c_20,
        ('celeba', 'Non-IID'): cn_20,
        ('cifar',  'IID'):     ci_20,
        ('cifar',  'Non-IID'): cin_20,
    }

    dataset_code = {
        'adult':  'A',
        'celeba': 'C',
        'cifar':  'CI',
        'imdb':   'I',
    }
    partition_code = {'IID': 'I', 'Non-IID': 'N'}

    for (dataset, partition), df in plot_data.items():
        fig, ax = plt.subplots(figsize=(5, 4))

        # loss, acc, fairDP, fairEO, rel, res, priv (whichever exist)
        pairs = contrib_columns(df, method, include_fair=True)
        value_vars = [col for _, col in pairs]
        palette = [BOX_COLORS[key] for key, _ in pairs]

        df_melted = df.melt(
            id_vars=['client_id'],
            value_vars=value_vars,
            var_name='type',
            value_name='value'
        )

        sns.boxplot(
            data=df_melted,
            x='client_id',
            y='value',
            hue='type',
            hue_order=value_vars,
            ax=ax,
            palette=palette,
            showfliers=False,
            legend=False
        )

        # Color box borders + whisker/cap/median lines so shrunk boxes stay visible
        color_box_borders(ax)

        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_ylabel("Metric", fontsize=10)
        ax.set_xlabel("Client", fontsize=10)
        ax.tick_params(axis='x', labelsize=8)
        unique_clients = sorted(df_melted['client_id'].unique())
        for i in range(len(unique_clients) - 1):
            ax.axvline(i + 0.5, color='lightgray',
                      linestyle='--', linewidth=0.8, zorder=0)

        plt.tight_layout()

        if 'num_clients' in df.columns:
            num_clients = int(df['num_clients'].iloc[0])
        else:
            num_clients = 4

        d_code = dataset_code.get(dataset, dataset[0].upper())
        p_code = partition_code[partition]
        filename = f"plots/S_{d_code}_{num_clients}_{p_code}_{method}.png"

        plt.savefig(filename, dpi=300)
        plt.show()
        plt.close(fig)


In [ ]:
def generate_heatmaps(method):
    MODE="last_round"  # "all_rows" | "seed_mean" | "last_round" | "round_avg"
    clients_list=(4, 20)
    partitions=("iid", "noniid")


    dataset_code = {"adult": "A", "celeba": "C", "cifar": "CI", "imdb": "I"}

    partition_code = {"iid": "I", "noniid": "N"}

    colmap = {
        "loss": f"{method}_loss_contribution",
        "acc": "acc_contribution",
        "fair": f"{method}_fair_contribution",
        "fair_eo": f"{method}_fair_eo_contribution",
        "rel":  f"{method}_rob_contribution",
        "res":  f"{method}_adv_contribution",
        "priv": f"{method}_priv_contribution",
    }

    # Tick-label display names (internal keys -> printed labels).
    # Only fair / fair_eo are renamed per the figure spec.
    display_labels = {
        "loss": "loss",
        "acc": "acc",
        "fair": "fairDP",
        "fair_eo": "fairEO",
        "rel": "rel",
        "res": "res",
        "priv": "priv",
    }

    pattern = f"results_*_{method}.csv"

    def _infer_dataset_from_filename(fp: str) -> str:
        m = re.search(r"results_(.+?)_" + re.escape(method) + r"\.csv$", os.path.basename(fp))
        return (m.group(1).lower() if m else "")

    def get_keys(sub: "pd.DataFrame"):
        # IMDB-nél nincs fair -> 4×4
        fair_col = colmap["fair"]
        if (fair_col in sub.columns) and (not sub[fair_col].isna().all()):
            return ["loss", "acc", "fair", "fair_eo", "rel", "res", "priv"]
        return ["loss", "acc", "rel", "res", "priv"]

    def drop_allnan_rows(df: "pd.DataFrame", keys):
        cols = [colmap[k] for k in keys]
        return df.dropna(subset=cols)

    def corr_on(df: "pd.DataFrame", keys):
        cols = [colmap[k] for k in keys]
        x = df[cols].copy()
        mat = x.corr(method="spearman")
        mat.index = keys
        mat.columns = keys
        return mat

    def corr_round_avg(sub: "pd.DataFrame", keys):
        mats = []
        cols = [colmap[k] for k in keys]

        for r in sorted(sub["round"].unique()):
            sr = sub[sub["round"] == r].groupby("client_id")[cols].mean()
            if sr.shape[0] < 3:
                continue
            mats.append(sr.corr(method="spearman").to_numpy())

        if not mats:
            return None

        avg = np.nanmean(np.stack(mats, axis=0), axis=0)
        return pd.DataFrame(avg, index=keys, columns=keys)

    def plot_heatmap(mat: "pd.DataFrame", save_path: str):
        # rename fair -> fairDP, fair_eo -> fairEO on the printed axes
        mat = mat.rename(index=display_labels, columns=display_labels)

        fig, ax = plt.subplots(figsize=(5.8, 5.0))

        rot = 45 if len(mat.columns) >= 5 else 0

        hm = sns.heatmap(
            mat,
            ax=ax,
            annot=True,
            fmt=".1f",
            annot_kws={"size": 16},
            vmin=-1, vmax=1, center=0,
            cmap="RdBu_r",
            square=True,
            linewidths=0.8,
            cbar=True,
            cbar_kws={"ticks": [-1, -0.5, 0, 0.5, 1]},
        )

        ax.xaxis.tick_top()
        ax.xaxis.set_label_position("top")
        ax.tick_params(axis="x", top=True, bottom=False, labeltop=True, labelbottom=False, pad=6)

        ax.set_xticklabels(ax.get_xticklabels(), rotation=rot, fontsize=18)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=rot, fontsize=18)

        cbar = hm.collections[0].colorbar
        cbar.ax.tick_params(labelsize=16, width=2.0, length=6)
        cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        fig.tight_layout()
        fig.savefig(save_path, bbox_inches="tight")
        plt.show()
        plt.close(fig)


    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Nem találok '{pattern}' fájlokat a working directory-ban.")

    dfs = []
    for fp in files:
        d = pd.read_csv(fp)

        if "dataset" not in d.columns:
            d["dataset"] = _infer_dataset_from_filename(fp)
        d["dataset"] = d["dataset"].astype(str).str.strip().str.lower()

        if "partition" not in d.columns:
            d["partition"] = np.where(d["dataset"].str.contains("noniid"), "noniid", "iid")
        d["partition"] = d["partition"].astype(str).str.strip().str.lower()

        dfs.append(d)

    df = pd.concat(dfs, ignore_index=True)

    saved = []

    for ds in sorted(df["dataset"].unique()):
        for part in partitions:
            for nc in clients_list:
                sub = df[
                    (df["dataset"] == ds) &
                    (df["partition"] == part) &
                    (df["num_clients"] == nc)
                ].copy()
                if sub.empty:
                    continue

                keys = get_keys(sub)
                cols = [colmap[k] for k in keys]

                if MODE == "all_rows":
                    work = drop_allnan_rows(sub, keys)
                    mat = corr_on(work, keys)

                elif MODE == "seed_mean":
                    work = sub.groupby(["client_id", "round"], as_index=False)[cols].mean()
                    work = drop_allnan_rows(work, keys)
                    mat = corr_on(work, keys)

                elif MODE == "last_round":
                    rmax = sub["round"].max()
                    work = sub[sub["round"] == rmax].groupby("client_id", as_index=False)[cols].mean()
                    work = drop_allnan_rows(work, keys)
                    mat = corr_on(work, keys)

                elif MODE == "round_avg":
                    mat = corr_round_avg(sub, keys)
                    if mat is None:
                        continue

                else:
                    raise ValueError("Ismeretlen MODE. Választható: all_rows | seed_mean | last_round | round_avg")

                base = ds.replace("noniid", "")
                fname = f"H_{nc}_{dataset_code.get(base, base)}_{partition_code[part]}_{method}.png"
                save_path = os.path.join("plots",fname)

                plot_heatmap(mat, save_path)
                saved.append(save_path)

    return saved


In [ ]:
def distribution_scores(method):
    df = df_gtg if method == 'gtg' else df_loo

    clients_list = [4, 20]

    datasets_by_partition = {
        "IID":     ["adult", "celeba", "cifar", "imdb"],
        "Non-IID": ["adultnoniid", "celebanoniid", "cifarnoniid", "imdbnoniid"],
    }

    dataset_pretty_base = {"adult": "A", "celeba": "C", "cifar": "CI", "imdb": "I"}

    partition_code = {"IID": "I", "Non-IID": "N"}
    partition_value = {"IID": "iid", "Non-IID": "noniid"}

    ZOOM_ABS_Q = 0.98
    ZOOM_PAD = 0.05
    MIN_LIM = 1e-6

    # --- Per-panel extra x-zoom (tighter limits) ---------------------------
    # "Zoom slightly more": multiply the auto limit by ZOOM_EXTRA (< 1).
    ZOOM_EXTRA = 0.7
    # Fig 4 (K=20): every panel EXCEPT (c) adult/non-IID/LOO and (l) imdb/non-IID/GTG.
    NO_EXTRA_ZOOM_20 = {("adult", "noniid", "l1o"), ("imdb", "noniid", "gtg")}
    # Fig 8 (K=4): only panels (a, b, d, f, h).
    EXTRA_ZOOM_4 = {
        ("adult",  "iid",    "l1o"),  # (a)
        ("adult",  "iid",    "gtg"),  # (b)
        ("adult",  "noniid", "gtg"),  # (d)
        ("celeba", "iid",    "gtg"),  # (f)
        ("celeba", "noniid", "gtg"),  # (h)
    }

    def wants_extra_zoom(num_clients, base, part_val, method):
        return True   # zoom every panel to the readable g/k level (user request)

    def base_dataset(ds: str) -> str:
        return ds.replace("noniid", "")

    for num_clients in clients_list:
        for partition, ds_list in datasets_by_partition.items():
            part_val = partition_value[partition]

            for ds in ds_list:
                base = base_dataset(ds).lower()

                df_filtered = df[
                    (df["num_clients"] == num_clients) &
                    (df["partition"] == part_val) &
                    (df["dataset"] == base)
                ].copy()

                dataset_name = dataset_pretty_base.get(base, base[0].upper())
                is_imdb = (base == "imdb")

                # loss, acc, (fairDP, fairEO), rel, res, priv — canonical order
                pairs = contrib_columns(df_filtered, method, include_fair=not is_imdb)

                series_by_metric = []  # (key, values) in plot order
                for key, col in pairs:
                    vals = df_filtered[col].dropna()
                    if vals.empty:
                        print(f"No valid values for {col} in dataset={base}, clients={num_clients}, skipping metric {key}.")
                        continue
                    series_by_metric.append((key, vals))

                if not series_by_metric:
                    print(f"No plottable metrics for dataset={base}, partition={part_val}, clients={num_clients}, skipping plot.")
                    continue

                fig, ax = plt.subplots(figsize=(5, 4))

                for key, vals in series_by_metric:
                    sns.kdeplot(
                        data=vals,
                        ax=ax,
                        linewidth=1.6,
                        fill=True,
                        alpha=0.3,
                        color=BOX_COLORS[key],
                    )

                ax.axvline(x=0.0, color="gray", linestyle=":", linewidth=1)
                ax.set_xlabel("Metric", fontsize=16)
                ax.set_ylabel("Density", fontsize=16)
                ax.tick_params(axis="both", labelsize=14)
                ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))

                leg = ax.get_legend()
                if leg is not None:
                    leg.remove()

                all_vals = np.concatenate([v.to_numpy() for _, v in series_by_metric])
                lim = np.quantile(np.abs(all_vals), ZOOM_ABS_Q)
                lim = max(lim, MIN_LIM) * (1.0 + ZOOM_PAD)
                if wants_extra_zoom(num_clients, base, part_val, method):
                    lim *= ZOOM_EXTRA
                ax.set_xlim(-lim, lim)

                plt.tight_layout()

                p_code = partition_code[partition]
                filename = f"plots/SD_{num_clients}_{dataset_name}_{p_code}_{method}.png"
                fig.savefig(filename, dpi=300)
                plt.show()
                plt.close(fig)
